# Bab 14 - Learning to Write Like Shakespeare: Long Short-Term Memory

**Nama:** Nanda Pratama ITM  
**NIM:** 101032330212  
**Kelas:** TK 47 01


> Notebook ini merupakan reproduksi edukatif dan pendalaman teori berdasarkan *Grokking Deep Learning* (Andrew W. Trask, Manning, 2019). Kode inti diadaptasi dari buku dan repositori pendamping resmi; perubahan kompatibilitas dijelaskan dalam dokumentasi.


## Capaian pembelajaran

Setelah menyelesaikan notebook ini, pembaca diharapkan mampu menjelaskan konsep utama bab, menghubungkannya dengan operasi NumPy, menjalankan contoh secara berurutan, dan menyebutkan batas interpretasinya.


## Character language modeling

Model karakter memprediksi karakter berikut dari konteks sebelumnya. Kosakata kecil membuat contoh mudah dibangun, tetapi sequence panjang menuntut penyimpanan konteks dan training yang stabil.


## Truncated BPTT

Truncated BPTT memotong sequence menjadi jendela dan melakukan backprop hanya dalam jendela tersebut. Ini menurunkan biaya, tetapi membatasi dependensi yang menerima gradien langsung.


## Vanishing dan exploding gradients

Perkalian Jacobian berulang dapat mengecilkan atau membesarkan gradien secara eksponensial. Gradient clipping mengendalikan ledakan, tetapi tidak menyelesaikan gradien yang menghilang.


## LSTM gates

LSTM memiliki forget, input, output gate, dan candidate state. Cell state menyediakan jalur aditif sehingga informasi dan gradien dapat bertahan lebih lama. Gate sigmoid mengontrol proporsi informasi, sedangkan kandidat biasanya memakai tanh.


## Evaluasi bahasa

Contoh teks yang tampak masuk akal bukan satu-satunya ukuran. Loss/perplexity pada data validasi, keragaman sampel, temperature, dan pemeriksaan memorization perlu dilaporkan.


## Reproduksi kode dan eksperimen

Jalankan sel dari atas ke bawah. `MODE_CEPAT=True` menjaga waktu eksekusi singkat; ubah ke `False` untuk konfigurasi yang lebih panjang.


In [1]:
from pathlib import Path
import numpy as np

np.set_printoptions(precision=4, suppress=True)
np.random.seed(1)
DATA_DIR = (Path.cwd().parent / "data").resolve()
MODE_CEPAT = True  # ubah ke False untuk eksperimen lebih panjang
assert DATA_DIR.is_dir(), f"Folder data tidak ditemukan: {DATA_DIR}"
print("Data:", DATA_DIR)
print("Mode cepat:", MODE_CEPAT)

def sigmoid(x):
    x = np.clip(x, -60, 60)
    return 1 / (1 + np.exp(-x))

def softmax(x):
    x = np.asarray(x)
    shifted = x - np.max(x, axis=-1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=-1, keepdims=True)


Data: D:\Kuliah\DL\UPLOAD_GITHUB\Grokking-Deep-Learning\data
Mode cepat: True


In [2]:
text = (DATA_DIR / "shakespear.txt").read_text(encoding="utf-8")
chars = sorted(set(text))
char2index = {ch: i for i, ch in enumerate(chars)}
rng = np.random.default_rng(14)
hidden = 32
embedding = rng.normal(0, 0.1, (len(chars), hidden))
w_forget = rng.normal(0, 0.1, (hidden * 2, hidden))
w_input = rng.normal(0, 0.1, (hidden * 2, hidden))
w_output = rng.normal(0, 0.1, (hidden * 2, hidden))
w_candidate = rng.normal(0, 0.1, (hidden * 2, hidden))

def lstm_step(token, h, c):
    joined = np.concatenate([embedding[token], h])
    forget = sigmoid(joined @ w_forget)
    input_gate = sigmoid(joined @ w_input)
    output_gate = sigmoid(joined @ w_output)
    candidate = np.tanh(joined @ w_candidate)
    c = forget * c + input_gate * candidate
    h = output_gate * np.tanh(c)
    return h, c, (forget, input_gate, output_gate)

h = np.zeros(hidden); c = np.zeros(hidden)
gate_means = []
for ch in text[:200]:
    h, c, gates = lstm_step(char2index[ch], h, c)
    gate_means.append([g.mean() for g in gates])
print("chars:", len(chars), "state norms:", np.linalg.norm(h), np.linalg.norm(c))
print("mean forget/input/output gates:", np.mean(gate_means, axis=0))


chars: 62 state norms: 0.11156928822886795 0.22355671289949822
mean forget/input/output gates: [0.4992 0.4999 0.4998]


In [3]:
# Ilustrasi exploding gradient dan gradient clipping
gradient = np.ones(8)
norms = []
for _ in range(20):
    gradient = gradient @ (np.eye(8) * 1.25)
    norms.append(np.linalg.norm(gradient))
clip_limit = 5.0
clipped = gradient * min(1.0, clip_limit / (np.linalg.norm(gradient) + 1e-12))
print("norm awal/akhir:", norms[0], norms[-1], "setelah clipping:", np.linalg.norm(clipped))


norm awal/akhir: 3.5355339059327378 245.32694666933986 setelah clipping: 4.99999999999998


## Interpretasi dan kesimpulan

Hasil harus dibaca bersama asumsi, konfigurasi, seed, serta data yang digunakan. Keberhasilan menjalankan kode tidak membuktikan generalisasi, kausalitas, keamanan, atau kelayakan deployment. Bandingkan hasil dengan teori di atas dan jelaskan setiap perbedaan sebelum mengubah konfigurasi.
